Imports

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import copy
from torchsummary import summary

Get Dataset Paths

In [12]:
train_dir = r"C:\SUTD\50.021 Artificial Intelligence\pppppp\Master_Dataset\train"
val_dir = r"C:\SUTD\50.021 Artificial Intelligence\pppppp\Master_Dataset\val"
test_dir = r"C:\SUTD\50.021 Artificial Intelligence\pppppp\Master_Dataset\test"

In [13]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    print(f'Using GPU: {torch.cuda.get_device_name(0)}')
else:
    print('Using CPU')

Using GPU: NVIDIA GeForce RTX 4060 Laptop GPU


Transforms

In [14]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


DataLoaders

In [15]:
# Load datasets
train_dataset = datasets.ImageFolder(train_dir, transform=transform)
val_dataset = datasets.ImageFolder(val_dir, transform=transform)
test_dataset = datasets.ImageFolder(test_dir, transform=transform)

print("Classes:", train_dataset.classes)
print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))
print("Test size:", len(test_dataset))

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

Classes: ['NORMAL', 'PNEUMONIA']
Train size: 14054
Val size: 1757
Test size: 1757


In [16]:
# Sanity check one sample
sample, label = train_dataset[0]
print("Sample shape:", sample.shape)
print("Label index:", label)
print("Class name:", train_dataset.classes[label])

Sample shape: torch.Size([3, 224, 224])
Label index: 0
Class name: NORMAL


Evaluation Function for all Models

In [17]:
def evaluate_model(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0

    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)

            # In eval mode, GoogLeNet usually returns main logits only
            if hasattr(outputs, "logits"):
                outputs = outputs.logits
            elif isinstance(outputs, tuple):
                outputs = outputs[0]

            loss = criterion(outputs, labels)
            running_loss += loss.item() * inputs.size(0)

            probs = torch.softmax(outputs, dim=1)[:, 1]
            preds = torch.argmax(outputs, dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs)

    return epoch_loss, acc, f1, auc

GOOGLENET MODEL

In [18]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchsummary import summary

googlenetmodel = models.googlenet(pretrained=True)

for param in googlenetmodel.parameters():
    param.requires_grad = False

googlenetmodel.fc = nn.Sequential(
    nn.Linear(googlenetmodel.fc.in_features, 512),
    nn.Dropout(p=0.3),
    nn.ReLU(inplace=True),
    nn.Linear(512, 128),
    nn.ReLU(inplace=True),
    nn.Linear(128, 32),
    nn.ReLU(inplace=True),
    nn.Linear(32, 2)
)

for name, param in googlenetmodel.named_parameters():
    if "inception5" in name:
        param.requires_grad = True

for param in googlenetmodel.fc.parameters():
    param.requires_grad = True


# Chuyển mô hình sang thiết bị (CPU hoặc GPU)
googlenetmodel = googlenetmodel.to(device)

# In ra cấu trúc mô hình
print(summary(googlenetmodel, (3, 224, 224)))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 112, 112]           9,408
       BatchNorm2d-2         [-1, 64, 112, 112]             128
       BasicConv2d-3         [-1, 64, 112, 112]               0
         MaxPool2d-4           [-1, 64, 56, 56]               0
            Conv2d-5           [-1, 64, 56, 56]           4,096
       BatchNorm2d-6           [-1, 64, 56, 56]             128
       BasicConv2d-7           [-1, 64, 56, 56]               0
            Conv2d-8          [-1, 192, 56, 56]         110,592
       BatchNorm2d-9          [-1, 192, 56, 56]             384
      BasicConv2d-10          [-1, 192, 56, 56]               0
        MaxPool2d-11          [-1, 192, 28, 28]               0
           Conv2d-12           [-1, 64, 28, 28]          12,288
      BatchNorm2d-13           [-1, 64, 28, 28]             128
      BasicConv2d-14           [-1, 64,

c:\Users\thoai\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\thoai\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=GoogLeNet_Weights.IMAGENET1K_V1`. You can also use `weights=GoogLeNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Googlenet Training parameters

In [19]:
from torch.optim.lr_scheduler import ReduceLROnPlateau
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(googlenetmodel.parameters(), lr=0.0001)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=4)

Training Loop for Googlenet

In [20]:
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

best_val_loss = 100.0
best_model_params = None

num_epochs = 15

for epoch in range(num_epochs):
    googlenetmodel.train()
    running_loss = 0.0
    total_train = 0
    correct_train = 0

    with tqdm(total=len(train_loader), desc=f'Epoch [{epoch+1}/{num_epochs}]') as pbar:
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()

            outputs = googlenetmodel(inputs)

            # ✅ Handle GoogLeNet outputs
            if hasattr(outputs, "logits"):
                outputs = outputs.logits
            elif isinstance(outputs, tuple):
                outputs = outputs[0]

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

            _, predicted = torch.max(outputs, 1)
            correct_train += (predicted == labels).sum().item()
            total_train += labels.size(0)

            pbar.update(1)
            pbar.set_postfix({'Loss': running_loss / total_train})

    epoch_loss = running_loss / len(train_dataset)
    train_acc = correct_train / total_train

    # ================= VALIDATION =================
    googlenetmodel.eval()

    val_loss = 0.0
    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = googlenetmodel(inputs)

            # ✅ Handle GoogLeNet outputs
            if hasattr(outputs, "logits"):
                outputs = outputs.logits
            elif isinstance(outputs, tuple):
                outputs = outputs[0]

            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)

            probs = torch.softmax(outputs, dim=1)[:, 1]
            preds = torch.argmax(outputs, dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    val_loss = val_loss / len(val_dataset)

    # ✅ Metrics
    val_acc = accuracy_score(all_labels, all_preds)
    val_f1 = f1_score(all_labels, all_preds)
    val_auc = roc_auc_score(all_labels, all_probs)

    print(
        f'Epoch [{epoch+1}/{num_epochs}] | '
        f'Train Loss: {epoch_loss:.4f} | '
        f'Train Acc: {train_acc:.4f} | '
        f'Val Loss: {val_loss:.4f} | '
        f'Val Acc: {val_acc:.4f} | '
        f'Val F1: {val_f1:.4f} | '
        f'Val AUC: {val_auc:.4f}'
    )

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_params = googlenetmodel.state_dict()

    scheduler.step(val_loss)


# ================= LOAD BEST MODEL =================
if best_model_params is not None:
    googlenetmodel.load_state_dict(best_model_params)

Epoch [1/15]: 100%|██████████| 440/440 [00:19<00:00, 22.35it/s, Loss=0.512]


Epoch [1/15] | Train Loss: 0.5118 | Train Acc: 0.7473 | Val Loss: 0.4482 | Val Acc: 0.7883 | Val F1: 0.8036 | Val AUC: 0.8719


Epoch [2/15]: 100%|██████████| 440/440 [00:19<00:00, 22.61it/s, Loss=0.42] 


Epoch [2/15] | Train Loss: 0.4200 | Train Acc: 0.8063 | Val Loss: 0.4452 | Val Acc: 0.7888 | Val F1: 0.7929 | Val AUC: 0.8742


Epoch [3/15]: 100%|██████████| 440/440 [00:19<00:00, 23.07it/s, Loss=0.348]


...

Epoch [13/15] | Train Loss: 0.0128 | Train Acc: 0.9959 | Val Loss: 1.1216 | Val Acc: 0.7615 | Val F1: 0.7604 | Val AUC: 0.8504


Epoch [14/15]: 100%|██████████| 440/440 [00:20<00:00, 21.74it/s, Loss=0.0121]


Epoch [14/15] | Train Loss: 0.0121 | Train Acc: 0.9964 | Val Loss: 1.1259 | Val Acc: 0.7655 | Val F1: 0.7719 | Val AUC: 0.8524


Epoch [15/15]: 100%|██████████| 440/440 [00:20<00:00, 21.87it/s, Loss=0.011]  


Epoch [15/15] | Train Loss: 0.0110 | Train Acc: 0.9969 | Val Loss: 1.1408 | Val Acc: 0.7661 | Val F1: 0.7700 | Val AUC: 0.8509


Save Model 

In [21]:
torch.save(googlenetmodel.state_dict(), "googlenet_finetuned_baseline.pth")
print("Model saved.")

Model saved.


ALEXNET MODEL

In [22]:
alexnetmodel = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)

alexnetmodel.classifier = nn.Sequential(
    nn.Dropout(),
    nn.Linear(9216, 4096),  
    nn.ReLU(inplace=True),
    nn.Dropout(),
    nn.Linear(4096, 1024),  
    nn.ReLU(inplace=True),
    nn.Linear(1024, 512), 
    nn.ReLU(inplace=True),
    nn.Linear(512, 128), 
    nn.ReLU(inplace=True),
    nn.Linear(128, 32),
    nn.ReLU(inplace=True),
    nn.Linear(32, 2)
)

for param in alexnetmodel.parameters():
    param.requires_grad = False

for param in alexnetmodel.classifier.parameters():
    param.requires_grad = True

alexnetmodel.to(device)
print(summary(alexnetmodel, (3, 224, 224)))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 64, 55, 55]          23,296
              ReLU-2           [-1, 64, 55, 55]               0
         MaxPool2d-3           [-1, 64, 27, 27]               0
            Conv2d-4          [-1, 192, 27, 27]         307,392
              ReLU-5          [-1, 192, 27, 27]               0
         MaxPool2d-6          [-1, 192, 13, 13]               0
            Conv2d-7          [-1, 384, 13, 13]         663,936
              ReLU-8          [-1, 384, 13, 13]               0
            Conv2d-9          [-1, 256, 13, 13]         884,992
             ReLU-10          [-1, 256, 13, 13]               0
           Conv2d-11          [-1, 256, 13, 13]         590,080
             ReLU-12          [-1, 256, 13, 13]               0
        MaxPool2d-13            [-1, 256, 6, 6]               0
AdaptiveAvgPool2d-14            [-1, 25

Alexnet parameters

In [23]:
from torch.optim.lr_scheduler import ReduceLROnPlateau
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(alexnetmodel.parameters(), lr=0.0001)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=4)

Training loop for Alexnet

In [24]:
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

best_val_loss = 100.0
best_model_params = None

num_epochs = 15

for epoch in range(num_epochs):
    alexnetmodel.train()
    running_loss = 0.0
    total_train = 0
    correct_train = 0

    with tqdm(total=len(train_loader), desc=f'Epoch [{epoch+1}/{num_epochs}]') as pbar:
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()

            outputs = alexnetmodel(inputs)

            # ✅ Handle GoogLeNet outputs
            if hasattr(outputs, "logits"):
                outputs = outputs.logits
            elif isinstance(outputs, tuple):
                outputs = outputs[0]

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

            _, predicted = torch.max(outputs, 1)
            correct_train += (predicted == labels).sum().item()
            total_train += labels.size(0)

            pbar.update(1)
            pbar.set_postfix({'Loss': running_loss / total_train})

    epoch_loss = running_loss / len(train_dataset)
    train_acc = correct_train / total_train

    # ================= VALIDATION =================
    alexnetmodel.eval()

    val_loss = 0.0
    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = alexnetmodel(inputs)

            # ✅ Handle GoogLeNet outputs
            if hasattr(outputs, "logits"):
                outputs = outputs.logits
            elif isinstance(outputs, tuple):
                outputs = outputs[0]

            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)

            probs = torch.softmax(outputs, dim=1)[:, 1]
            preds = torch.argmax(outputs, dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    val_loss = val_loss / len(val_dataset)

    # ✅ Metrics
    val_acc = accuracy_score(all_labels, all_preds)
    val_f1 = f1_score(all_labels, all_preds)
    val_auc = roc_auc_score(all_labels, all_probs)

    print(
        f'Epoch [{epoch+1}/{num_epochs}] | '
        f'Train Loss: {epoch_loss:.4f} | '
        f'Train Acc: {train_acc:.4f} | '
        f'Val Loss: {val_loss:.4f} | '
        f'Val Acc: {val_acc:.4f} | '
        f'Val F1: {val_f1:.4f} | '
        f'Val AUC: {val_auc:.4f}'
    )

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_params = alexnetmodel.state_dict()

    scheduler.step(val_loss)


# ================= LOAD BEST MODEL =================
if best_model_params is not None:
    alexnetmodel.load_state_dict(best_model_params)

Epoch [1/15]: 100%|██████████| 440/440 [00:17<00:00, 25.25it/s, Loss=0.505]


Epoch [1/15] | Train Loss: 0.5053 | Train Acc: 0.7517 | Val Loss: 0.4387 | Val Acc: 0.7826 | Val F1: 0.7817 | Val AUC: 0.8760


Epoch [2/15]: 100%|██████████| 440/440 [00:17<00:00, 24.70it/s, Loss=0.452]


Epoch [2/15] | Train Loss: 0.4516 | Train Acc: 0.7821 | Val Loss: 0.4298 | Val Acc: 0.7883 | Val F1: 0.8017 | Val AUC: 0.8825


Epoch [3/15]: 100%|██████████| 440/440 [00:17<00:00, 24.52it/s, Loss=0.442]


...

Epoch [13/15] | Train Loss: 0.3609 | Train Acc: 0.8262 | Val Loss: 0.4105 | Val Acc: 0.8071 | Val F1: 0.7934 | Val AUC: 0.8956


Epoch [14/15]: 100%|██████████| 440/440 [00:16<00:00, 26.33it/s, Loss=0.351]


Epoch [14/15] | Train Loss: 0.3513 | Train Acc: 0.8317 | Val Loss: 0.4121 | Val Acc: 0.7997 | Val F1: 0.8044 | Val AUC: 0.8953


Epoch [15/15]: 100%|██████████| 440/440 [00:16<00:00, 26.67it/s, Loss=0.341]


Epoch [15/15] | Train Loss: 0.3406 | Train Acc: 0.8418 | Val Loss: 0.4131 | Val Acc: 0.7917 | Val F1: 0.8047 | Val AUC: 0.8905


Save model

In [25]:
torch.save(alexnetmodel.state_dict(), "alexnet_finetuned_baseline.pth")
print("Model saved.")

Model saved.


RESNET-18 MODEL

In [26]:
resnet18model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

resnet18model.fc = nn.Sequential(
    nn.Dropout(),
    nn.Linear(512, 128),
    nn.ReLU(inplace=True),
    nn.Linear(128, 32),
    nn.ReLU(inplace=True),
    nn.Linear(32, 2),
    nn.ReLU(inplace=True)
)

# freeze everything
for param in resnet18model.parameters():
    param.requires_grad = False

# unfreeze final layer
for param in resnet18model.fc.parameters():
    param.requires_grad = True

resnet18model.to(device)
print(summary(resnet18model, (3, 224, 224)))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 112, 112]           9,408
       BatchNorm2d-2         [-1, 64, 112, 112]             128
              ReLU-3         [-1, 64, 112, 112]               0
         MaxPool2d-4           [-1, 64, 56, 56]               0
            Conv2d-5           [-1, 64, 56, 56]          36,864
       BatchNorm2d-6           [-1, 64, 56, 56]             128
              ReLU-7           [-1, 64, 56, 56]               0
            Conv2d-8           [-1, 64, 56, 56]          36,864
       BatchNorm2d-9           [-1, 64, 56, 56]             128
             ReLU-10           [-1, 64, 56, 56]               0
       BasicBlock-11           [-1, 64, 56, 56]               0
           Conv2d-12           [-1, 64, 56, 56]          36,864
      BatchNorm2d-13           [-1, 64, 56, 56]             128
             ReLU-14           [-1, 64,

Resnet 18 parameters

In [27]:
from torch.optim.lr_scheduler import ReduceLROnPlateau
criterion = nn.CrossEntropyLoss()
# Định nghĩa optimizer và scheduler
optimizer = optim.Adam(resnet18model.parameters(), lr=0.0001)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=4)

Training loop for resnet-18

In [28]:
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

best_val_loss = 100.0
best_model_params = None

num_epochs = 15

for epoch in range(num_epochs):
    resnet18model.train()
    running_loss = 0.0
    total_train = 0
    correct_train = 0

    with tqdm(total=len(train_loader), desc=f'Epoch [{epoch+1}/{num_epochs}]') as pbar:
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()

            outputs = resnet18model(inputs)

            # ✅ Handle GoogLeNet outputs
            if hasattr(outputs, "logits"):
                outputs = outputs.logits
            elif isinstance(outputs, tuple):
                outputs = outputs[0]

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

            _, predicted = torch.max(outputs, 1)
            correct_train += (predicted == labels).sum().item()
            total_train += labels.size(0)

            pbar.update(1)
            pbar.set_postfix({'Loss': running_loss / total_train})

    epoch_loss = running_loss / len(train_dataset)
    train_acc = correct_train / total_train

    # ================= VALIDATION =================
    resnet18model.eval()

    val_loss = 0.0
    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = resnet18model(inputs)

            # ✅ Handle GoogLeNet outputs
            if hasattr(outputs, "logits"):
                outputs = outputs.logits
            elif isinstance(outputs, tuple):
                outputs = outputs[0]

            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)

            probs = torch.softmax(outputs, dim=1)[:, 1]
            preds = torch.argmax(outputs, dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    val_loss = val_loss / len(val_dataset)

    # ✅ Metrics
    val_acc = accuracy_score(all_labels, all_preds)
    val_f1 = f1_score(all_labels, all_preds)
    val_auc = roc_auc_score(all_labels, all_probs)

    print(
        f'Epoch [{epoch+1}/{num_epochs}] | '
        f'Train Loss: {epoch_loss:.4f} | '
        f'Train Acc: {train_acc:.4f} | '
        f'Val Loss: {val_loss:.4f} | '
        f'Val Acc: {val_acc:.4f} | '
        f'Val F1: {val_f1:.4f} | '
        f'Val AUC: {val_auc:.4f}'
    )

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_params = resnet18model.state_dict()

    scheduler.step(val_loss)


# ================= LOAD BEST MODEL =================
if best_model_params is not None:
    resnet18model.load_state_dict(best_model_params)


Epoch [1/15]: 100%|██████████| 440/440 [00:16<00:00, 26.36it/s, Loss=0.654]


Epoch [1/15] | Train Loss: 0.6540 | Train Acc: 0.6532 | Val Loss: 0.6258 | Val Acc: 0.7376 | Val F1: 0.7544 | Val AUC: 0.7788


Epoch [2/15]: 100%|██████████| 440/440 [00:17<00:00, 25.61it/s, Loss=0.632]


Epoch [2/15] | Train Loss: 0.6320 | Train Acc: 0.7023 | Val Loss: 0.6235 | Val Acc: 0.7399 | Val F1: 0.7335 | Val AUC: 0.7621


Epoch [3/15]: 100%|██████████| 440/440 [00:16<00:00, 26.23it/s, Loss=0.629]


...

Epoch [13/15] | Train Loss: 0.6150 | Train Acc: 0.7256 | Val Loss: 0.6033 | Val Acc: 0.7501 | Val F1: 0.7623 | Val AUC: 0.8004


Epoch [14/15]: 100%|██████████| 440/440 [00:16<00:00, 26.00it/s, Loss=0.614]


Epoch [14/15] | Train Loss: 0.6141 | Train Acc: 0.7280 | Val Loss: 0.6065 | Val Acc: 0.7524 | Val F1: 0.7440 | Val AUC: 0.7805


Epoch [15/15]: 100%|██████████| 440/440 [00:16<00:00, 26.54it/s, Loss=0.613]


Epoch [15/15] | Train Loss: 0.6129 | Train Acc: 0.7261 | Val Loss: 0.6040 | Val Acc: 0.7604 | Val F1: 0.7545 | Val AUC: 0.7870


Save model

In [29]:
torch.save(resnet18model.state_dict(), "resnet18_finetuned_baseline.pth")
print("Model saved.")

Model saved.
